# 05 - Baseline evaluation

This notebook evaluates all baseline models on both experimental tasks:

- HDFS log anomaly classification;
- QA response quality evaluation.

Results are saved to `artifacts/results/baselines/`.

In [ ]:
import exp_common as E
import pandas as pd

RUN_SEEDS = [42, 123, 777]
EVAL_LOGS_N = 2000
EVAL_QA_N = 500
BATCH_LOGS = 16
BATCH_QA = 4
JUDGE_N = 100

logs_eval = E.read_jsonl(E.PREPARED_DIR / "logs_test_eval.jsonl")[:EVAL_LOGS_N]
qa_eval = E.read_jsonl(E.PREPARED_DIR / "qa_test_eval.jsonl")[:EVAL_QA_N]
out_dir = E.RESULTS_DIR / "baselines"
out_dir.mkdir(parents=True, exist_ok=True)

def eval_one(name, adapter_dir=None, base_adapter_dir=None, seed="none"):
    print("Evaluating", name, seed)
    lm = E.evaluate_logs(adapter_dir, logs_eval, out_dir / f"{name}_seed_{seed}_logs.json", batch_size=BATCH_LOGS, base_adapter_dir=base_adapter_dir)
    qm = E.evaluate_qa(adapter_dir, qa_eval, out_dir / f"{name}_seed_{seed}_qa.json", batch_size=BATCH_QA, judge_sample=JUDGE_N, base_adapter_dir=base_adapter_dir)
    return {
        "model": name, "seed": seed,
        **{f"logs_{k}": v for k, v in lm.items() if not isinstance(v, (list, dict))},
        **{f"qa_{k}": v for k, v in qm.items() if not isinstance(v, (list, dict))},
    }

rows = [eval_one("base", None, None, "none")]
for seed in RUN_SEEDS:
    base = E.ADAPTERS_DIR
    cpt = base / "cpt" / f"seed_{seed}" / "adapter"
    candidates = {
        "cpt": (cpt, None),
        "ft_logs": (base / "logs" / f"seed_{seed}" / "adapter", None),
        "ft_qa": (base / "qa" / f"seed_{seed}" / "adapter", None),
        "multitask_lora": (base / "multitask" / f"seed_{seed}" / "adapter", None),
        "cpt_ft_logs": (base / "logs_on_cpt" / f"seed_{seed}" / "adapter", cpt),
        "cpt_ft_qa": (base / "qa_on_cpt" / f"seed_{seed}" / "adapter", cpt),
        "multitask_lora_on_cpt": (base / "multitask_on_cpt" / f"seed_{seed}" / "adapter", cpt),
    }
    for name, (adapter, anchor) in candidates.items():
        if (adapter / "adapter_model.safetensors").exists():
            rows.append(eval_one(name, adapter, anchor, seed))
        else:
            print("Missing, skip:", adapter)

df = pd.DataFrame(rows)
df.to_csv(out_dir / "baseline_metrics_raw.csv", index=False)
num_cols = [c for c in df.columns if c not in ["model", "seed"] and pd.api.types.is_numeric_dtype(df[c])]
df.groupby("model")[num_cols].agg(["mean", "std"]).to_csv(out_dir / "baseline_metrics_mean_std.csv")
df